# Music Streaming Analysis

**Felipe Duarte · TripleTen Data Science Bootcamp, Sprint 2**

Do music playback volumes vary by city and day of the week? This analysis compares Springfield and Shelbyville on Monday, Wednesday, and Friday using Python and pandas. Each row represents a recorded playback event.

## Load and inspect the data

Download the original CSV using the instructions in the README and place it in the local `data` directory. Inspect the dataset dimensions and count missing values without displaying individual playback records.

In [1]:
from pathlib import Path

import pandas as pd

data_path = Path('data/music_project_en.csv')
if not data_path.is_file():
    raise FileNotFoundError(f'Missing {data_path}. See README.md for download instructions.')

df = pd.read_csv(data_path)
print(f'{len(df):,} playback records across {df.shape[1]} columns')
print(df.isna().sum().loc[lambda counts: counts > 0].to_string())

65,079 playback records across 7 columns
Track     1343
artist    7567
genre     1198


## Clean the playback records

Standardize the column names, label missing track, artist, and genre values as `unknown`, remove exact duplicates, and merge alternate spellings of hip-hop. Missing genre labels can affect genre comparisons. The analysis below counts playback events using city and day.

In [2]:
df.columns = df.columns.str.strip().str.lower()
df = df.rename(columns={'userid': 'user_id'})

metadata_columns = ['track', 'artist', 'genre']
df[metadata_columns] = df[metadata_columns].fillna('unknown')

duplicate_count = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
df['genre'] = df['genre'].replace(['hip', 'hop', 'hip-hop'], 'hiphop')

print(f'Exact duplicates removed: {duplicate_count:,}')
print(f'Playback records after cleaning: {len(df):,}')
print(f'Missing values after cleaning: {df.isna().sum().sum():,}')

Exact duplicates removed: 3,826
Playback records after cleaning: 61,253
Missing values after cleaning: 0


## Compare playback counts

Filter by day and city, then count the playback records. A user who played several tracks contributes one record per playback.

In [3]:
def number_tracks(day, city):
    tracks_by_day = df[df['day'] == day]
    tracks_by_city = tracks_by_day[tracks_by_day['city'] == city]
    return tracks_by_city['user_id'].count()


days = ['Monday', 'Wednesday', 'Friday']
cities = ['Springfield', 'Shelbyville']

playbacks = pd.DataFrame(
    {city: [number_tracks(day, city) for day in days] for city in cities},
    index=pd.Index(days, name='Day'),
)
playbacks

,Springfield,Shelbyville
Day,,
Monday,15740,5614
Wednesday,11056,7003
Friday,15945,5895


## Findings

Springfield recorded more playbacks on all three days and peaked on Friday (15,945). Shelbyville peaked on Wednesday (7,003), the same day Springfield recorded its lowest count (11,056).

These results support the hypothesis that playback activity varies by city and day **within the dataset**. The comparison is descriptive and covers three observed weekdays. Differences in the number of active users could affect aggregate counts. A per-user analysis would require dividing by the number of distinct active users in each group.